# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

The baseline is intentionally transparent. A page ranks higher for review when it combines:

1. **Visibility** — more impressions means the page matters enough to review.
2. **Freshness risk** — a longer time since update increases review priority.
3. **Position opportunity** — visible pages closer to page one get extra priority.
4. **Depth gap** — relatively thin pages with real visibility get a small boost.

The baseline does **not** use `trend_direction`, `trend_pct`, or the proxy label to score a page. Those are reserved only for evaluation.

Reason codes are likewise feature-only:
- `stale_visible`
- `page_one_opportunity`
- `low_ctr_visible`
- `thin_visible`
- `low_engagement_visible`
- `general_review`

A human can read the rule, challenge it, and compare later models against it.

In [ ]:
import os, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root = find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root = Path(REPO_DIR).resolve()
os.chdir(root)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_proxy"] = df["trend_direction"].str.lower().eq("down").astype(int)

def pct_rank(s):
    return pd.Series(s).rank(pct=True, method="average").fillna(0)

visibility = pct_rank(np.log1p(df["impressions_90d"].clip(lower=0)))
freshness = pct_rank(df["days_since_last_update"].fillna(0))
pos = df["avg_position"].fillna(0)
position_opportunity = ((51 - pos.clip(lower=1, upper=50)) / 50) * visibility * (pos > 0)
depth_gap = (1 - pct_rank(df["word_count"].fillna(df["word_count"].median()))) * visibility

df["baseline_score"] = (
    0.40*visibility + 0.30*freshness + 0.25*position_opportunity + 0.05*depth_gap
).clip(0,1)

def reasons(row):
    r=[]
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500: r.append("stale_visible")
    if 0 < row["avg_position"] <= 10 and row["impressions_90d"] >= 500: r.append("page_one_opportunity")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5: r.append("low_ctr_visible")
    if pd.notna(row["word_count"]) and 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250: r.append("thin_visible")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30) or
        (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ): r.append("low_engagement_visible")
    return "|".join(r or ["general_review"])

df["reason_codes"] = df.apply(reasons, axis=1)

print("Rows scored:", len(df))
print("Proxy base rate:", round(df["is_declining_proxy"].mean(),3))
print("Baseline formula uses target-derived fields:", False)


## 2. Build the ranked queue (writes the CSV)

The full queue is ranked by the feature-only `baseline_score`. The CSV is intentionally written under `work/outputs/`, which the repository's leak guard excludes from git. The notebook is the reproducible receipt.

For model comparison later, I freeze one client-holdout split with seed 42. The baseline's Precision@K is evaluated only on held-out clients.

In [ ]:
from pathlib import Path

rng = np.random.default_rng(42)
clients = df["client_id"].drop_duplicates().to_numpy()
test_client_count = max(1, int(round(len(clients)*0.20)))
test_clients = set(rng.permutation(clients)[:test_client_count])
df["split"] = np.where(df["client_id"].isin(test_clients), "test", "train")

queue = df.sort_values(["baseline_score","impressions_90d"], ascending=[False,False]).copy()
queue["baseline_rank"] = np.arange(1, len(queue)+1)

out_cols = [
    "baseline_rank","content_id","client_id","baseline_score","reason_codes",
    "impressions_90d","avg_position","ctr","content_age_days","days_since_last_update",
    "sessions_90d","engagement_rate","scroll_rate","word_count","is_declining_proxy","split"
]
Path("work/outputs").mkdir(parents=True, exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

def precision_at_k(frame,k):
    top=frame.sort_values("baseline_score",ascending=False).head(k)
    return float(top["is_declining_proxy"].mean())

test = df[df["split"]=="test"].copy()
metrics = {
    "seed":42,
    "train_rows":int((df["split"]=="train").sum()),
    "test_rows":int((df["split"]=="test").sum()),
    "train_clients":int(df.loc[df["split"]=="train","client_id"].nunique()),
    "test_clients":int(df.loc[df["split"]=="test","client_id"].nunique()),
    "base_rate_test":float(test["is_declining_proxy"].mean()),
    "precision_at_20":precision_at_k(test,20),
    "precision_at_50":precision_at_k(test,50),
    "precision_at_100":precision_at_k(test,100)
}
Path("work/outputs/w04_baseline_metrics.json").write_text(json.dumps(metrics,indent=2))

print(json.dumps(metrics,indent=2))
print("Queue written: work/outputs/baseline_action_score.csv")


## 3. Top-20 review

The review below is a **human audit of the ranking logic**, not a claim that these pages should be edited automatically.

For every top item I attach:
- the proposed review action,
- feature-only reason codes,
- a confidence note based on how many transparent signals agree,
- and a concrete condition that could make the recommendation wrong.

The target proxy is shown only after ranking so I can inspect whether the rule is working; it never generates the recommendation.

In [ ]:
def action_for(row):
    rs=set(str(row["reason_codes"]).split("|"))
    if "low_ctr_visible" in rs: return "review_title_meta_and_intent"
    if "thin_visible" in rs: return "review_depth_and_coverage"
    if "stale_visible" in rs: return "review_for_refresh"
    if "low_engagement_visible" in rs: return "review_engagement_and_intent"
    return "monitor_and_review_context"

def wrong_if(row):
    rs=set(str(row["reason_codes"]).split("|"))
    if "stale_visible" in rs: return "content is intentionally evergreen or recently improved outside captured metadata"
    if "low_ctr_visible" in rs: return "SERP layout or query mix explains CTR rather than snippet quality"
    if "thin_visible" in rs: return "short format fully satisfies intent"
    return "traffic is seasonal, consolidated to a sibling page, or too noisy"

top20=queue.head(20).copy()
top20["action"]=top20.apply(action_for,axis=1)
top20["signal_count"]=top20["reason_codes"].str.count(r"\|")+1
top20["confidence_note"]=np.where(top20["signal_count"]>=2,"multiple transparent signals agree","single-signal candidate; review cautiously")
top20["what_would_make_it_wrong"]=top20.apply(wrong_if,axis=1)

review=top20[[
    "baseline_rank","baseline_score","action","reason_codes","confidence_note",
    "what_would_make_it_wrong","is_declining_proxy"
]]
pd.set_option("display.max_colwidth",90)
print(review.to_string(index=False))
print("\nTop-20 proxy precision:", round(top20["is_declining_proxy"].mean(),3))


## 4. Weak picks + leakage check

A useful baseline should expose its weak spots. I flag top-20 rows with only a generic or single reason as weaker picks. I also explicitly check that none of the target-derived fields, product decisions, IDs-as-features, or future outcomes are used in the score.

The rule is frozen here. Week 5 must beat it on the **same client holdout and the same Precision@K metrics**; I will not tune this rule after seeing the model result.

In [ ]:
scoring_features = [
    "impressions_90d","days_since_last_update","avg_position","word_count",
    "ctr","sessions_90d","engagement_rate","scroll_rate"
]
forbidden = {"trend_direction","trend_pct","is_declining_proxy","content_id","client_id"}

weak = top20[top20["signal_count"]<=1][
    ["baseline_rank","baseline_score","reason_codes","action","is_declining_proxy"]
]
print("Weak/single-signal top-20 picks:", len(weak))
print(weak.to_string(index=False) if len(weak) else "None")

print("\nForbidden scoring fields present:", sorted(forbidden.intersection(scoring_features)))
assert forbidden.isdisjoint(scoring_features)

leakage_receipt = {
    "forbidden_fields_in_score": sorted(forbidden.intersection(scoring_features)),
    "target_used_for_scoring": False,
    "ids_used_as_features": False,
    "future_window_used": False,
    "baseline_frozen": True
}
Path("work/outputs/w04_leakage_receipt.json").write_text(json.dumps(leakage_receipt,indent=2))
print(json.dumps(leakage_receipt,indent=2))


## Self-check

- [x] The rule is stated in plain language and encoded transparently
- [x] Reason codes use only observable, feature-time signals
- [ ] Notebook executed top to bottom with visible outputs
- [x] Ranked queue writes to `work/outputs/baseline_action_score.csv`
- [x] Top-20 human review includes action, reason, confidence, and failure condition
- [x] Baseline is frozen for Week 5 comparison
- [x] No client names, raw URLs, queries, credentials, or target-derived scoring fields are used
- [ ] Submit the public repository URL on the ML-07 card